# Agentic Log Anomaly Explanation Pipeline

## Complete Walkthrough

This notebook provides a clean, end-to-end walkthrough of the **Screener-Reasoner** pipeline for generating traceable explanations of log anomalies.

### Pipeline Architecture

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Log Data   │ ──▶ │  Screener   │ ──▶ │  Retriever  │ ──▶ │     LLM     │
│  (Sessions) │     │  (Detect)   │     │    (RAG)    │     │  (Explain)  │
└─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
                           │                   │                   │
                           ▼                   ▼                   ▼
                    Anomaly Prob        Evidence Hits       Explanation
                    + Margin            (4 anomaly +        + Claims
                                        1 normal)           + Evidence IDs
```

### Components

| Component | Purpose | Implementation |
|-----------|---------|----------------|
| **Data Loader** | Load and split log sessions | `BGLDataLoader`, `HDFSDataLoader` |
| **Screener** | Detect anomalies (binary classification) | AllLinLog neural model |
| **Evidence Store** | Index training sessions for retrieval | BM25 corpus |
| **Retriever** | Find similar historical sessions | BM25 with mixed retrieval |
| **Prompt Builder** | Format prompt with evidence | Structured JSON schema |
| **LLM Client** | Generate explanations | Ollama (llama3.1:8b) |
| **Verifier** | Validate explanation faithfulness | Rule-based checks |

## 1. Setup & Imports

In [ ]:
# Standard library
import sys
import json
import time
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.data_loader import BGLDataLoader, Session
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore
from src.retriever import BM25Retriever, Retriever
from src.signature_generator import SignatureGenerator
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, format_evidence_block
from src.llm_client import LLMClient
from src.verifier import Verifier

print("✓ All imports successful")

## 2. Data Loading

Load the BGL (Blue Gene/L) supercomputer log dataset.

- **Train split**: Used to build evidence store (never seen during inference)
- **Test split**: Sessions to detect and explain anomalies

In [ ]:
# Load BGL dataset
loader = BGLDataLoader(log_file="../logs/BGL.log")
loader.load()
loader.print_stats()

# Get splits
train_sessions = loader.get_train()
test_sessions = loader.get_test()

print(f"\nTrain sessions: {len(train_sessions):,}")
print(f"Test sessions: {len(test_sessions):,}")

# Count anomalies in each split
train_anomaly = sum(1 for s in train_sessions if s.label == 1)
test_anomaly = sum(1 for s in test_sessions if s.label == 1)
print(f"\nTrain anomalies: {train_anomaly:,} ({train_anomaly/len(train_sessions):.1%})")
print(f"Test anomalies: {test_anomaly:,} ({test_anomaly/len(test_sessions):.1%})")

### Example Session

Each session is a group of log lines with a label (0=normal, 1=anomaly).

In [ ]:
# Show an example anomalous session
example_anomaly = next(s for s in test_sessions if s.label == 1)

print(f"Session ID: {example_anomaly.session_id}")
print(f"Label: {'ANOMALY' if example_anomaly.label == 1 else 'NORMAL'}")
print(f"Lines: {len(example_anomaly.lines)}")
print("\nFirst 5 lines:")
for i, line in enumerate(example_anomaly.lines[:5], 1):
    print(f"  {i}. {line[:100]}..." if len(line) > 100 else f"  {i}. {line}")

## 3. Screener (Anomaly Detection)

The Screener is a neural network (AllLinLog) that classifies log sessions as normal or anomalous.

**Output:**
- `pred`: Binary prediction (0=normal, 1=anomaly)
- `prob`: Probability distribution [P(normal), P(anomaly)]
- `margin`: Confidence measure (|P(anomaly) - P(normal)|)

In [ ]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="BGL",
    model_path="../best_model/best_model_20250724_072857.pth"
)

print(f"Model parameters: {screener.model_params:,}")

In [ ]:
# Screen a sample of test sessions
sample_size = 100
sample_sessions = test_sessions[:sample_size]

print(f"Screening {sample_size} sessions...")
start = time.time()
screener_outputs = screener.screen_sessions(sample_sessions)
elapsed = time.time() - start

print(f"Done in {elapsed:.2f}s ({elapsed/sample_size*1000:.1f}ms per session)")

# Count predictions
predicted_anomalies = [o for o in screener_outputs if o.is_anomaly]
print(f"\nPredicted anomalies: {len(predicted_anomalies)} / {sample_size}")

In [ ]:
# Show a screener output example
if predicted_anomalies:
    output = predicted_anomalies[0]
    print("Example Screener Output:")
    print(f"  Session ID: {output.session_id}")
    print(f"  Prediction: {'ANOMALY' if output.is_anomaly else 'NORMAL'}")
    print(f"  Anomaly Probability: {output.anomaly_prob:.2%}")
    print(f"  Confidence Margin: {output.margin:.4f}")

## 4. Evidence Store

The Evidence Store indexes all training sessions for RAG retrieval.

**Key features:**
- Stores normalized text for each session
- Tracks label (normal/anomaly) for mixed retrieval
- Supports multiple evidence types (session, signature)

In [ ]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="BGL")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

### Add Signature Cards

Signature cards provide domain knowledge about common error patterns.

In [ ]:
# Generate error signature cards from training anomalies
sig_generator = SignatureGenerator()
train_anomalies = [s for s in train_sessions if s.label == 1]
signatures = sig_generator.generate_signatures(train_anomalies)

print(f"Generated {len(signatures)} signature cards:")
for sig in signatures:
    print(f"  - {sig.name}: {sig.frequency:,} matches")

# Add signatures to evidence store
evidence_store.add_signatures(signatures)
print(f"\nEvidence store now has {len(evidence_store.documents):,} documents")

## 5. Retriever (RAG)

The Retriever finds similar historical sessions using BM25.

### Mixed Retrieval (Phase 2)

To support **contrast claims** ("unlike normal sessions..."), we retrieve:
- **4 anomaly** sessions (for pattern matching)
- **1 normal** session (for contrast)

In [ ]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
retriever.build_index()

In [ ]:
# Test mixed retrieval on an anomalous session
test_session = sample_sessions[screener_outputs.index(predicted_anomalies[0])]

# Standard retrieval (top-5 any label)
print("=== Standard Retrieval (top-5 any) ===")
standard_hits = retriever.retrieve_for_session(test_session, top_k=5)
for h in standard_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

# Mixed retrieval (4 anomaly + 1 normal)
print("\n=== Mixed Retrieval (4 anomaly + 1 normal) ===")
mixed_hits = retriever.retrieve_for_session_mixed(test_session, top_k_anomaly=4, top_k_normal=1)
for h in mixed_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

## 6. Prompt Builder

The Prompt Builder formats the session and evidence into a structured prompt.

### Evidence ID Convention
- `[E0]` = Query session (the session being analyzed)
- `[E1]`, `[E2]`, ... = Retrieved historical evidence

### Claim Types
| Type | Description | Example |
|------|-------------|----------|
| `observation` | Direct observation from E0 | "E0 contains KERNEL FATAL errors" |
| `pattern_match` | Matches known anomaly | "E0 pattern matches E1, E2" |
| `contrast` | Differs from normal | "Unlike E5 (normal), E0 shows FATAL" |

In [ ]:
# Initialize prompt builder
builder = PromptBuilder(
    max_log_lines=20,
    max_chars_per_evidence=500,
    max_evidence_items=5
)

# Create a mock screener output for the test session
scr_output = predicted_anomalies[0]

# Build prompt
system_prompt, user_prompt = builder.build_prompt(
    session=test_session,
    screener_output=scr_output,
    evidence_hits=mixed_hits
)

print("=== SYSTEM PROMPT (truncated) ===")
print(system_prompt[:500])
print("...")

print("\n=== USER PROMPT (truncated) ===")
print(user_prompt[:1500])
print("...")

## 7. LLM Client

Call the LLM (Ollama with llama3.1:8b) to generate an explanation.

In [ ]:
# Initialize LLM client
llm_client = LLMClient(
    provider="ollama",
    model="llama3.1:8b",
    temperature=0.1,
    max_tokens=1024,
    timeout=120
)

# Check availability
if llm_client.is_available():
    print(f"✓ LLM ({llm_client.model}) is available")
else:
    print(f"✗ LLM not available. Start with: ollama serve")

In [ ]:
# Generate explanation
print("Generating explanation...")
start = time.time()

response = llm_client.generate(
    prompt=user_prompt,
    system_prompt=system_prompt,
    json_mode=True
)

elapsed = time.time() - start
print(f"Done in {elapsed:.2f}s")
print(f"Tokens: {response.total_tokens}")

In [ ]:
# Parse and display the explanation
explanation_dict = json.loads(response.content)

print("=" * 60)
print("LLM EXPLANATION")
print("=" * 60)
print(f"\nPrediction: {explanation_dict.get('prediction')}")
print(f"\nSummary: {explanation_dict.get('summary')}")
print(f"\nClaims ({len(explanation_dict.get('claims', []))}):\n")

for i, claim in enumerate(explanation_dict.get('claims', []), 1):
    claim_type = claim.get('type', 'N/A')
    evidence_ids = claim.get('evidence_ids', [])
    print(f"  [{i}] ({claim_type})")
    print(f"      Claim: {claim.get('claim', 'N/A')}")
    print(f"      Evidence: {evidence_ids}")
    print()

## 8. Verifier

The Verifier checks that the explanation is **faithful** to the evidence.

### Verification Tiers

| Tier | Name | Checks |
|------|------|--------|
| **Tier-1** | Format & Citation Validity | JSON parseable, required fields, valid evidence IDs |
| **Tier-2** | Evidence Coverage | ≥80% of claims cite evidence |

### Not Verified (Limitations)
- Tier-3: Semantic support (claim text matches evidence)
- Tier-4: Factual correctness
- Human evaluation

In [ ]:
# Initialize verifier
verifier = Verifier()

# Convert dict to TraceExplanation
trace_exp = TraceExplanation(
    prediction=explanation_dict.get('prediction'),
    summary=explanation_dict.get('summary'),
    claims=[Claim(
        type=c.get('type', 'observation'),
        claim=c.get('claim'),
        evidence_ids=c.get('evidence_ids', [])
    ) for c in explanation_dict.get('claims', [])],
    insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
)

# Build evidence ID mapping
evidence_id_mapping = builder.build_evidence_id_mapping(test_session, mixed_hits)

# Verify
verification = verifier.verify(
    explanation=trace_exp,
    evidence_hits=mixed_hits,
    evidence_id_mapping=evidence_id_mapping
)

print("=" * 60)
print("VERIFICATION RESULT")
print("=" * 60)
print(f"\nPassed: {verification.passed}")
print(f"Total checks: {verification.total_checks}")
print(f"Passed checks: {verification.passed_checks}")
print(f"Failed checks: {verification.failed_checks}")

## 9. Complete Pipeline Function

Putting it all together in a single function.

In [ ]:
def explain_session(
    session: Session,
    screener_output: ScreenerOutput,
    retriever: BM25Retriever,
    builder: PromptBuilder,
    llm_client: LLMClient,
    verifier: Verifier,
    signatures: list = None
) -> dict:
    """
    Generate and verify an explanation for an anomalous session.
    
    Returns a dict with explanation, verification, and metrics.
    """
    start = time.time()
    
    # 1. Retrieve evidence (mixed: 4 anomaly + 1 normal)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, 
        top_k_anomaly=4, 
        top_k_normal=1
    )
    
    # 2. Build prompt
    system_prompt, user_prompt = builder.build_prompt(
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits
    )
    
    # 3. Call LLM
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    # 4. Parse response
    try:
        explanation_dict = json.loads(response.content)
        parse_success = True
    except json.JSONDecodeError:
        explanation_dict = {"prediction": "anomaly", "summary": "Parse error", "claims": []}
        parse_success = False
    
    # 5. Convert to TraceExplanation
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # 6. Verify
    evidence_id_mapping = builder.build_evidence_id_mapping(session, evidence_hits)
    verification = verifier.verify(trace_exp, evidence_hits, evidence_id_mapping)
    
    elapsed = time.time() - start
    
    return {
        'session_id': session.session_id,
        'explanation': explanation_dict,
        'verification_passed': verification.passed,
        'parse_success': parse_success,
        'tokens': response.total_tokens,
        'latency_ms': elapsed * 1000
    }

print("✓ explain_session function defined")

In [ ]:
# Test the complete pipeline function
result = explain_session(
    session=test_session,
    screener_output=scr_output,
    retriever=retriever,
    builder=builder,
    llm_client=llm_client,
    verifier=verifier
)

print("=" * 60)
print("PIPELINE RESULT")
print("=" * 60)
print(f"Session: {result['session_id']}")
print(f"Parse success: {result['parse_success']}")
print(f"Verification: {'PASSED ✓' if result['verification_passed'] else 'FAILED ✗'}")
print(f"Tokens: {result['tokens']}")
print(f"Latency: {result['latency_ms']:.0f}ms")

## 10. Batch Processing Example

Process multiple sessions and collect metrics.

In [ ]:
# Process a small batch of anomalies
from tqdm import tqdm

batch_size = 5  # Small batch for demo
batch_results = []

# Get sessions and outputs for predicted anomalies
anomaly_pairs = [
    (sample_sessions[i], screener_outputs[i])
    for i, o in enumerate(screener_outputs)
    if o.is_anomaly
][:batch_size]

print(f"Processing {len(anomaly_pairs)} sessions...")

for session, scr_output in tqdm(anomaly_pairs, desc="Explaining"):
    result = explain_session(
        session=session,
        screener_output=scr_output,
        retriever=retriever,
        builder=builder,
        llm_client=llm_client,
        verifier=verifier
    )
    batch_results.append(result)

# Summary
print("\n" + "=" * 60)
print("BATCH SUMMARY")
print("=" * 60)

passed = sum(1 for r in batch_results if r['verification_passed'])
total_tokens = sum(r['tokens'] for r in batch_results)
avg_latency = sum(r['latency_ms'] for r in batch_results) / len(batch_results)

print(f"\nTotal sessions: {len(batch_results)}")
print(f"Verification passed: {passed} / {len(batch_results)} ({passed/len(batch_results):.1%})")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/session: {total_tokens/len(batch_results):.0f}")
print(f"Avg latency: {avg_latency:.0f}ms")

## 11. Summary

### Pipeline A Results (Full BGL Dataset)

| Metric | Value |
|--------|-------|
| Dataset | BGL (Blue Gene/L Supercomputer) |
| Total Log Lines | 4,747,963 |
| Test Sessions | 71,221 |
| Predicted Anomalies | 5,849 (8.2%) |
| LLM | Llama 3.1:8b (local via Ollama) |
| Evidence Store | 332,358 documents |
| Retrieved Evidence | Top-5 (4 anomaly + 1 normal) |

### Verification Results

| Tier | Pass Rate |
|------|------------|
| Tier-1 (Format & Citation) | 100% (5849/5849) |
| Tier-2 (Evidence Coverage) | 100% (5849/5849) |

### Key Design Decisions

1. **Mixed Retrieval**: 4 anomaly + 1 normal evidence enables contrast claims
2. **E0 Convention**: Query session is always E0, retrieved evidence is E1-E5
3. **Typed Claims**: observation, pattern_match, contrast for traceability
4. **Rule-based Verification**: Fast, deterministic faithfulness checks

---

## Next Steps

1. **Full Pipeline Run**: Use `pipelines/explain_all.py` for complete dataset
2. **HDFS Dataset**: Apply same pipeline to HDFS logs
3. **Pipeline B**: Budgeted explanation (explain low-confidence cases first)
4. **Human Evaluation**: Sample explanations for quality review